In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False

import torch
assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'sentence-transformers', 'lightgbm',
    'scikit-learn', 'pandas', 'numpy', 'tqdm'], check=True)

sys.path.insert(0, f'{PROJECT_DIR}/src')
print(f'PROJECT_DIR = {PROJECT_DIR}')


In [ ]:
# ── Cell 2: Load Blind Test Data ─────────────────────────────────────────────
# Place blind test JSON in data/raw/blind_test.json
# Accepts both raw RFC-BENCH schema and pre-normalised schema.

from predict import load_blind_data

BLIND_INPUT  = f'{PROJECT_DIR}/data/raw/blind_test.json'
OUTPUT_CSV   = f'{PROJECT_DIR}/results/predictions/blind_predictions.csv'

texts, ids = load_blind_data(BLIND_INPUT)
print(f'Loaded {len(texts)} blind samples')
print(f'Sample: {texts[0][:200]}')


In [ ]:
# ── Cell 3: Tier 1 Features ───────────────────────────────────────────────────
import numpy as np

from features.tier1_nli       import extract_nli_feature_matrix
from features.tier1_embeddings import extract_all_embeddings, EmbeddingDistanceExtractor
from features.tier1_coherence  import extract_coherence_feature_matrix
from features.tier1_epistemic  import extract_epistemic_feature_matrix

print('--- NLI features ---')
t1_nli = extract_nli_feature_matrix(texts, device='cuda')

print('\n--- FinBERT embeddings ---')
embeddings = extract_all_embeddings(texts, device='cuda')

# Load fitted extractor from training — do NOT refit
extractor = EmbeddingDistanceExtractor.load(f'{PROJECT_DIR}/models/emb_extractor.pkl')
t1_emb = extractor.transform(embeddings)

print('\n--- Coherence features ---')
t1_coh = extract_coherence_feature_matrix(texts, device='cuda')

print('\n--- Epistemic features (CPU) ---')
t1_epi = extract_epistemic_feature_matrix(texts)

t1 = np.hstack([t1_nli, t1_emb, t1_coh, t1_epi])
print(f'\nTier 1 shape: {t1.shape}')
np.save(f'{PROJECT_DIR}/feature_cache/blind_tier1.npy', t1)
print('✅ Tier 1 saved.')


In [ ]:
# ── Cell 4: Tier 2 Features ───────────────────────────────────────────────────
# Uses saved fine-tuned models — direct inference (no OOF/CV)
import numpy as np
from features.tier2_encoder import inference_softmax_preds

print('--- FinBERT inference ---')
finbert_probs = inference_softmax_preds(
    texts,
    model_dir=f'{PROJECT_DIR}/models/finbert_finetuned',
    device='cuda',
)

print('\n--- DeBERTa inference ---')
deberta_probs = inference_softmax_preds(
    texts,
    model_dir=f'{PROJECT_DIR}/models/deberta_finetuned',
    device='cuda',
)

t2 = np.hstack([finbert_probs, deberta_probs])   # (N, 4)
print(f'\nTier 2 shape: {t2.shape}')
np.save(f'{PROJECT_DIR}/feature_cache/blind_tier2.npy', t2)
print('✅ Tier 2 saved.')


In [ ]:
# ── Cell 5: Tier 3 Features ───────────────────────────────────────────────────
import numpy as np
from features.tier3_perplexity import extract_perplexity_feature_matrix

print('--- MLM perplexity (slow ~2hrs on T4) ---')
t3 = extract_perplexity_feature_matrix(texts, device='cuda')
print(f'Tier 3 shape: {t3.shape}')
np.save(f'{PROJECT_DIR}/feature_cache/blind_tier3.npy', t3)
print('✅ Tier 3 saved.')


In [ ]:
# ── Cell 6: Meta-Classifier → Predictions + Metrics ─────────────────────────
import pickle, numpy as np, pandas as pd
from pathlib import Path

t1 = np.load(f'{PROJECT_DIR}/feature_cache/blind_tier1.npy')
t2 = np.load(f'{PROJECT_DIR}/feature_cache/blind_tier2.npy')
t3 = np.load(f'{PROJECT_DIR}/feature_cache/blind_tier3.npy')
X  = np.hstack([t1, t2, t3])
print(f'Combined feature matrix: {X.shape}')

with open(f'{PROJECT_DIR}/models/meta_model.pkl', 'rb') as f:
    meta_model = pickle.load(f)

preds      = meta_model.predict(X)
pred_probs = meta_model.predict_proba(X)

df = pd.DataFrame({
    'id':         ids,
    'prediction': preds,
    'label':      ['True' if p == 1 else 'False' for p in preds],
    'prob_true':  pred_probs[:, 1].round(4),
    'prob_false': pred_probs[:, 0].round(4),
})

Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f'✅ Predictions saved → {OUTPUT_CSV}')
print(f'   True : {(preds==1).sum()} ({(preds==1).mean():.1%})')
print(f'   False: {(preds==0).sum()} ({(preds==0).mean():.1%})')

# ── Metrics (only when ground-truth labels are available) ─────────────────────
# For the held-out test split, labels ARE available.
# For the official blind set, skip this block.
import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')
from predict import _load_records_for_metrics, _add_metrics

true_labels = [r.get('label') for r in _load_records_for_metrics(BLIND_INPUT)]
if all(l is not None for l in true_labels):
    import numpy as np
    df = _add_metrics(df, preds, pred_probs, true_labels, OUTPUT_CSV)
else:
    print('ℹ️  No ground-truth labels — metrics skipped (official blind set).')

df.head(10)
